In [ ]:
from snowflake.snowpark import Session
import pandas as pd
import numpy as np

In [24]:
# 1. Define your connection parameters
connection_parameters = {
    "account": "rt27428.eu-central-1",               # Wise Snowflake account
    "user": "priscaadenike.adeoti@transferwise.com",        # your Wise email
    "authenticator": "EXTERNALBROWSER",              # opens Okta in browser
    "warehouse": "ANALYSTS",
    "database": "ANALYTICS_DB",
}

# 2. Create a Snowflake session
session = Session.builder.configs(connection_parameters).create()


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://transferwise.okta-emea.com/app/snowflake/exk4istpb5gZUyV8u0i7/sso/saml?SAMLRequest=nZJPc9owEMW%2Fikc927INFKoBMhSGKZM0oWAyTG7CXohiWXK0MoZ%2B%2Bsr8ySSH5NCbR367v7f7tn9zKKS3B4NCqwGJgpB4oFKdCbUbkFUy9XvEQ8tVxqVWMCBHQHIz7CMvZMlGlX1WC3itAK3nGilkzY8BqYximqNApngByGzKlqPfdywOQsYRwViHI5eSDIVjPVtbMkrrug7qVqDNjsZhGNLwB3WqRvKNvEOUXzNKo61OtbyWHNxMnyAiGrYbhFM4wvxS%2BFOo8wq%2BomzOImS%2FkmTuzx%2BWCfFG1%2BnGWmFVgFmC2YsUVou7swF0DhZJ3G3HvQAqPwVlDZd%2BFKDS9VbyHFJdlJV1jQP3RbeQUal3wq1rNhmQMhdZa8r%2FvPDNZjJu718kx5TfPrzCul5v8%2F1huRa9v3Cbr6vagE6J93gNN27CnSFWMFNNpNY9hfF3P2z5cSeJYtbpsE4rCHvdJ%2BJNXKRCcXuqvPp2ThVuwdQCIdC55T4UwE82eVnStwkoHPK2QFtuOrun1fGxV4WiSxE1bbIj5%2FNhJytm%2BB9L6dP3DS7HeO%2FymU3mWor06E21Kbj9PL4oiE4vIvO3JymDggs5yjIDiC5GKXU9NsCtu3lrKiB0eKZ%2BvPrhPw%3D%3D&RelayState=ver%3A3-hint%3A11514986436542-ETMsDgAAAZ0lENP%2FABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEGKfjB943kYx7p30Z7Fa0KQAAACgornKxKZ%2

In [ ]:
# Last 6 months data to estimate average loss if we miss a bad merchant in any rule group
all_rules_query = """
WITH rule_mapping AS (
    SELECT 
        ID as case_id,
        REFERENCE_ID as profile_id,
        TIME_CREATED as created_time,
        CASE
            WHEN UPPER(metadata) LIKE UPPER('%PaymentForRequestRecentlyPublished%') THEN 'PaymentForRequestRecentlyPublished'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_UniqueCardsRefusedLast24HoursThresholdBreached_PauseMerchant%') THEN 'UniqueCardsRefusedLast24HoursThresholdBreached'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_UniqueCardsRefusedLast24HoursThresholdBreached_Reusable_PauseMerchant%')THEN 'UniqueCardsRefusedLast24HoursThresholdBreached_Reusable'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_UniqueCardsRefusedLast24HoursThresholdBreached_Wisetag_PauseMerchant%')THEN 'UniqueCardsRefusedLast24HoursThresholdBreached_Wisetag'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_emailAgeScoreAboveThreshold_ReviewMerchant%')THEN 'emailAgeScoreAboveThreshold'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_HighRiskIssuerRefusalReason_ReviewMerchant%')THEN 'HighRiskIssuerRefusalReason'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_PAYFAC_POST_AUTH_PAYFAC_SUB_MERCHANTAbuserFlag_AlertSlack%')THEN 'AbuserFlag'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_AverageDailyPayFacTransactionExceedsAverageDailyTransfer_PauseMerchant%')THEN 'AverageDailyPayFacTransactionExceedsAverageDailyTransfer'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_AttemptedAmountExceedsTransferAmount_PauseMerchant%') THEN 'AttemptedAmountExceedsTransferAmount' 
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_Bin6AttemptedVolumeSpike_ReviewMerchant%') THEN 'Bin6AttemptedVolumeSpike'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_ProtonmailEmailDomain_PauseMerchant%') THEN 'ProtonmailEmailDomain'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_IssuerAndMerchantConcentration_ReviewMerchant%') THEN 'IssuerAndMerchantConcentration'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_IssuerIcardADPattern_DeclineAndOffboard%') THEN 'IssuerIcardADPattern'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_LTVHigherThresholdBreached_PauseMerchant%') THEN 'LTVHigherThresholdBreached'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_LTVLowerThresholdBreached_ReviewMerchant%') THEN 'LTVLowerThresholdBreached'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_PayerDisputeThreshold_ReviewMerchant') THEN 'PayerDisputeThreshold'
            WHEN UPPER(metadata) LIKE UPPER('%Rule fired - o_WithdrawalRule_PauseMerchant') THEN 'WithdrawalRule'
            ELSE 'Transaction_amount'
        END AS rule_group
    FROM DDCASE.DD_CASE
    WHERE TIME_CREATED >= DATEADD(month, -6, CURRENT_DATE())
        AND case_type = 'ACQUIRING_MERCHANT_REVIEW'
        AND state = 'CLOSED'
),

rule_cases_deduplicated AS (
    -- Get the FIRST case for each profile-rule combination
    SELECT 
        rule_group,
        profile_id,
        MIN(created_time) as created_time
    FROM rule_mapping
    GROUP BY rule_group, profile_id
),

--payment information
merchant_payments AS (
    SELECT 
        rc.rule_group,
        rc.profile_id,
        p.CARD_ATTEMPT_ID,
        p.ATTEMPTED_GBP_AMOUNT_GROSS,
        p.FRAUD_FLAG,
        p.CHARGEBACK_AMOUNT_GBP,
        m.date_offboarded,
    FROM rule_cases_deduplicated rc
    LEFT JOIN RPT_PRODUCT.PAY_WITH_CARD_MERCHANTS m ON m.profile_id = rc.profile_id
    LEFT JOIN RPT_PRODUCT.PAY_WITH_CARD_PAYMENTS p ON rc.profile_id = p.PROFILE_ID
        AND p.ATTEMPT_TIME > rc.created_time
        AND p.ATTEMPT_TIME <= m.date_offboarded
        AND p.successful_transaction_flag = 'success'
    
)

SELECT 
    rule_group,
    COUNT(DISTINCT profile_id) as unique_merchants,
    COALESCE(COUNT(CARD_ATTEMPT_ID), 0)  as total_payments,
    COALESCE(SUM(CASE WHEN FRAUD_FLAG = TRUE THEN 1 ELSE 0 END), 0) as NOF_payments,
    COALESCE(SUM(CASE WHEN FRAUD_FLAG = TRUE THEN ATTEMPTED_GBP_AMOUNT_GROSS ELSE 0 END), 0) as NOF_amount_gbp,
    COALESCE(SUM(CASE WHEN CHARGEBACK_AMOUNT_GBP IS NOT NULL THEN 1 ELSE 0 END), 0) as chargeback_payments,
    COALESCE(SUM(COALESCE(CHARGEBACK_AMOUNT_GBP, 0)), 0) as chargeback_amount_gbp,
    COALESCE(
        COUNT(DISTINCT CASE WHEN CHARGEBACK_AMOUNT_GBP IS NOT NULL OR FRAUD_FLAG = TRUE THEN profile_id END),
        0
    ) as problematic_merchants,
    COALESCE(
        SUM(CASE WHEN FRAUD_FLAG = TRUE THEN ATTEMPTED_GBP_AMOUNT_GROSS ELSE 0 END), 0
    ) + COALESCE(
        SUM(COALESCE(CHARGEBACK_AMOUNT_GBP, 0)), 0
    ) as total_fraud_gbp,
    CASE 
        WHEN COUNT(DISTINCT CASE WHEN CHARGEBACK_AMOUNT_GBP IS NOT NULL OR FRAUD_FLAG = TRUE THEN profile_id END) > 0
        THEN
            (
                COALESCE(SUM(CASE WHEN FRAUD_FLAG = TRUE THEN ATTEMPTED_GBP_AMOUNT_GROSS ELSE 0 END), 0) +
                COALESCE(SUM(COALESCE(CHARGEBACK_AMOUNT_GBP, 0)), 0)
            )
            / NULLIF(
                COUNT(DISTINCT CASE WHEN CHARGEBACK_AMOUNT_GBP IS NOT NULL OR FRAUD_FLAG = TRUE THEN profile_id END),
                0
            )
        ELSE 0
END AS avg_fraud_per_problematic_merchant  -- This will give an idea o if we do not review a merchant by this rule and they turn out to be bad, how much we might lose on average
FROM merchant_payments
GROUP BY rule_group
ORDER BY total_fraud_gbp DESC
"""

print("=" * 100)
print("COMPREHENSIVE LOSS ANALYSIS FOR ALL RULES (Last 6 Months)")
print("=" * 100)
print("\n⏳ Executing query... This may take a moment...\n")

df_all_rules = session.sql(all_rules_query).to_pandas()

print("✓ Query complete!\n")
print("=" * 100)

COMPREHENSIVE LOSS ANALYSIS FOR ALL RULES (Last 6 Months)

⏳ Executing query... This may take a moment...

✓ Query complete!



In [ ]:
# Display the results in a clear format
print("=" * 100)
print("RESULTS: LOSS ESTIMATION BY RULE")
print("=" * 100 + "\n")

if len(df_all_rules) > 0:
    # Normalise column names to lowercase
    df_all_rules.columns = [c.lower() for c in df_all_rules.columns]

    # Format the dataframe for better display
    df_display = df_all_rules.copy()
    df_display['nof_amount_gbp'] = df_display['nof_amount_gbp'].apply(lambda x: f'£{x:,.2f}')
    df_display['chargeback_amount_gbp'] = df_display['chargeback_amount_gbp'].apply(lambda x: f'£{x:,.2f}')
    df_display['total_fraud_gbp'] = df_display['total_fraud_gbp'].apply(lambda x: f'£{x:,.2f}')
    df_display['avg_fraud_per_problematic_merchant'] = round(df_display['avg_fraud_per_problematic_merchant'], 2)
    
    print(df_display.to_string(index=False))
    
    print("\n" + "=" * 100)
    print("SUMMARY STATISTICS")
    print("=" * 100 + "\n")
    
    total_merchants = df_all_rules['unique_merchants'].sum()
    total_problematic_merchants = df_all_rules['problematic_merchants'].sum()
    total_payments = df_all_rules['total_payments'].sum()
    total_NOF_amt = df_all_rules['nof_amount_gbp'].sum()
    total_cb_amt = df_all_rules['chargeback_amount_gbp'].sum()
    total_fraud = df_all_rules['total_fraud_gbp'].sum()
    
    print(f"Total Unique Merchants: {total_merchants:,}")
    print(f"Total Payments Analyzed: {total_payments:,}")
    print(f"Total NOF Amount: £{total_NOF_amt:,.2f}")
    print(f"Total Chargeback Amount: £{total_cb_amt:,.2f}")
    print(f"TOTAL FRAUD (ALL RULES): £{total_fraud:,.2f}")
    print(f"Monthly Average Fraud: £{total_fraud/6:,.2f}")
    
    print("\n" + "=" * 100)
    
else:
    print("⚠️ No data returned for the specified rules")

print("\n✓ Results saved as 'df_all_rules'")

RESULTS: LOSS ESTIMATION BY RULE

                                              rule_group  unique_merchants  total_payments  nof_payments nof_amount_gbp  chargeback_payments chargeback_amount_gbp  problematic_merchants total_fraud_gbp  avg_fraud_per_problematic_merchant
                    AttemptedAmountExceedsTransferAmount               143              62             0          £0.00                    4            £45,978.38                      2      £45,978.38                            22989.19
                      PaymentForRequestRecentlyPublished                82              63             0          £0.00                    7            £44,661.92                      4      £44,661.92                            11165.48
AverageDailyPayFacTransactionExceedsAverageDailyTransfer               879            4627            38     £14,104.96                   64            £30,429.38                     37      £44,534.34                             1203.63
              

In [114]:
# Create the current state dataframe

Avg_Cases_Per_Month = [23, 292, 32, 10, 27, 1, 9, 67, 7, 14, 14, 10, 20, 6, 77, 13, 67]  # this will change when Xander updates the sheet
Avg_Handling_Time_Minutes = [20, 35, 23, 29, 26, 36, 110, 33, 19, 46, 21, 42, 38, 32, 32, 23, 16]  #this will change when Xander updates the sheet
Current_Hours_Per_Month = [  #calculating the current hours
    round((cases * aht) / 60,2)
    for cases, aht in zip(Avg_Cases_Per_Month, Avg_Handling_Time_Minutes)
]

# Input data
current_state = {
    'Rule_Name': [
        'AttemptedAmountExceedsTransferAmount',
        'AverageDailyPayFacTransactionExceedsAverageDailyTransfer',
        'Bin6AttemptedVolumeSpike',
        'HighRiskIssuerRefusalReason',
        'IssuerAndMerchantConcentration',
        'IssuerIcardADPattern',
        'LTVHigherThresholdBreached',
        'LTVLowerThresholdBreached',
        'PayerDisputeThreshold',
        'PaymentForRequestRecentlyPublished',
        'ProtonmailEmailDomain',
        'UniqueCardsRefusedLast24HoursThresholdBreached',
        'UniqueCardsRefusedLast24HoursThresholdBreached_Reusable',
        'UniqueCardsRefusedLast24HoursThresholdBreached_Wisetag',
        'WithdrawalRule',
        'emailAgeScoreAboveThreshold',
        'Transaction_amount',
        
    ],

    'rule_precision_L3M' : [67.82, 74.13, 71, 69.23, 44.32, 100, 30.77, 41.95, 94.44, 62.5, 58.82, 84.62, 78.95, 83.33, 55.74, 83.87, 50.67],
    'per_case_Precision_%': [52.17, 40.63, 29.03, 50.00, 25.00, 100.00, 0.00, 39.39, 0.00, 41.18, 42.86, 60.00, 38.46, 71.43, 40.12, 43.75, 13.04],
    'Avg_Cases_Per_Month': Avg_Cases_Per_Month,
    'Avg_Handling_Time_Minutes': Avg_Handling_Time_Minutes,
    'Current_Review_Type': ['Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Deep', 'Deep', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard', 'Standard'],
    'Current_Hours_Per_Month': Current_Hours_Per_Month
}

df_current = pd.DataFrame(current_state)



# Add loss data from our earlier analysis
 #first we define a helper function to get the average fraud per problematic merchant for each rule from our earlier analysis dataframe (df_display) as a value rather than series
def get_scalar(df, rule, col, default=0.0):
    s = df.loc[df["rule_group"] == rule, col]
    if s.empty:
        return default
    return float(s.iloc[0])  # numeric scalar


loss_mapping = {
    'AttemptedAmountExceedsTransferAmount': get_scalar(df_display, 'AttemptedAmountExceedsTransferAmount', 'avg_fraud_per_problematic_merchant'),
    'PaymentForRequestRecentlyPublished': get_scalar(df_display, 'PaymentForRequestRecentlyPublished', 'avg_fraud_per_problematic_merchant'),
    'AverageDailyPayFacTransactionExceedsAverageDailyTransfer': get_scalar(df_display, 'AverageDailyPayFacTransactionExceedsAverageDailyTransfer', 'avg_fraud_per_problematic_merchant'),
    'UniqueCardsRefusedLast24HoursThresholdBreached_Wisetag': get_scalar(df_display, 'UniqueCardsRefusedLast24HoursThresholdBreached_Wisetag', 'avg_fraud_per_problematic_merchant'),
    'Bin6AttemptedVolumeSpike': get_scalar(df_display, 'Bin6AttemptedVolumeSpike', 'avg_fraud_per_problematic_merchant'),
    'WithdrawalRule': get_scalar(df_display, 'WithdrawalRule', 'avg_fraud_per_problematic_merchant'),
    'UniqueCardsRefusedLast24HoursThresholdBreached': get_scalar(df_display, 'UniqueCardsRefusedLast24HoursThresholdBreached', 'avg_fraud_per_problematic_merchant'),
    'UniqueCardsRefusedLast24HoursThresholdBreached_Reusable': get_scalar(df_display, 'UniqueCardsRefusedLast24HoursThresholdBreached_Reusable', 'avg_fraud_per_problematic_merchant'),
    'emailAgeScoreAboveThreshold': get_scalar(df_display, 'emailAgeScoreAboveThreshold', 'avg_fraud_per_problematic_merchant'),
    'HighRiskIssuerRefusalReason': get_scalar(df_display, 'HighRiskIssuerRefusalReason', 'avg_fraud_per_problematic_merchant'),
    'ProtonmailEmailDomain': get_scalar(df_display, 'ProtonmailEmailDomain', 'avg_fraud_per_problematic_merchant'),
    'LTVLowerThresholdBreached': get_scalar(df_display, 'LTVLowerThresholdBreached', 'avg_fraud_per_problematic_merchant'),
    'IssuerAndMerchantConcentration': get_scalar(df_display, 'IssuerAndMerchantConcentration', 'avg_fraud_per_problematic_merchant'),
    'LTVHigherThresholdBreached': get_scalar(df_display, 'LTVHigherThresholdBreached', 'avg_fraud_per_problematic_merchant'),
    'PayerDisputeThreshold': get_scalar(df_display, 'PayerDisputeThreshold', 'avg_fraud_per_problematic_merchant'),
    'IssuerIcardADPattern': get_scalar(df_display, 'IssuerIcardADPattern', 'avg_fraud_per_problematic_merchant'),
    'Transaction_amount': get_scalar(df_display, 'Transaction_amount', 'avg_fraud_per_problematic_merchant'),
}


df_current['avg_fraud_volume_per_problematic_merchant'] = (df_current['Rule_Name'].map(loss_mapping))    
df_current['monthly_avg_fraud_volume_per_problematic_merchant'] = df_current['avg_fraud_volume_per_problematic_merchant'] / 6  #on average what is the loss per merchant per month

# Review time constants (in minutes)
LIGHT_REVIEW_MIN = 15 #Xander used this in the sheet instead of 12.4
STANDARD_REVIEW_MIN = 35.15
DEEP_REVIEW_MIN = 47.97

print("=" * 120)
print("CURRENT STATE ANALYSIS")
print("=" * 120 + "\n")

# Calculate current weekly hours
current_monthly_hours = df_current['Current_Hours_Per_Month'].sum()  
current_weekly_hours = current_monthly_hours * 12 / 52 + 20 # Adding 20 hours for NOF, CB
Target_Weekly_Hours = 75 #Enter the Capacity Hours / Week from the sheet (currently H34)
print(f"Current Total Monthly Hours: {current_monthly_hours:.2f}")
print(f"Current Weekly Hours: {current_weekly_hours:.2f}")
print(f"Target Weekly Hours: {Target_Weekly_Hours:.2f}")
print(f"Reduction Needed: {current_weekly_hours - Target_Weekly_Hours} hours/week ({(current_weekly_hours - Target_Weekly_Hours)/current_weekly_hours * 100:.1f}%)")

print("\n" + "=" * 120)

CURRENT STATE ANALYSIS

Current Total Monthly Hours: 365.39
Current Weekly Hours: 104.32
Target Weekly Hours: 75.00
Reduction Needed: 29.32076923076923 hours/week (28.1%)



### Scoring using per case precision calculated by Xander


In [122]:
# Create scoring system to determine most risky rule so it receives optimal review type
# Factors: Loss risk, Precision, Volume (cases * AHT)

def calculate_priority_score(row):
    """
    Calculate priority score based on:
    - Monthly loss (higher = higher priority)
    - per case precision (higher = higher confidence, but also means we're catching them)
    - Volume (higher volume = more impact)
    """
    
    # Normalize loss (0-100 scale) so there is not too much skew from the highest loss rule and we can still differentiate between the lower loss rules
    max_loss = df_current['monthly_avg_fraud_volume_per_problematic_merchant'].max()  # Max average monthlyloss per problematic merchant
    loss_score = min((row['monthly_avg_fraud_volume_per_problematic_merchant'] / max_loss) * 100, 100) if max_loss > 0 else 0
    
    # Per case precision score 
    precision_score = row['per_case_Precision_%']

    
    # Volume score (normalized as well)
    max_effort = (df_current['Avg_Cases_Per_Month'] * df_current['Avg_Handling_Time_Minutes']).max()
    effort = row['Avg_Cases_Per_Month'] * row['Avg_Handling_Time_Minutes']
    volume_score = (effort / max_effort) * 100 if max_effort > 0 else 0

    
    # Weighted score: Loss (50%), Precision (30%), Volume (20%)
    priority_score = (loss_score * 0.5) + (precision_score * 0.3) + (volume_score * 0.2)
    
    return priority_score

df_current['Priority_Score'] = round(df_current.apply(calculate_priority_score, axis=1), 2)

# Sort by priority
df_current_sorted = df_current.sort_values(by = 'Priority_Score', ascending=False)

print("\nRULE PRIORITIZATION")
print("=" * 120)

df_current_sorted[['Rule_Name', 'monthly_avg_fraud_volume_per_problematic_merchant', 'Avg_Cases_Per_Month', 'Avg_Handling_Time_Minutes', 'per_case_Precision_%', 'Current_Review_Type', 'Priority_Score']]


RULE PRIORITIZATION


,Rule_Name,monthly_avg_fraud_volume_per_problematic_merchant,Avg_Cases_Per_Month,Avg_Handling_Time_Minutes,per_case_Precision_%,Current_Review_Type,Priority_Score
0,AttemptedAmountExceedsTransferAmount,3831.531667,23,20,52.17,Standard,66.55
9,PaymentForRequestRecentlyPublished,1860.913333,14,46,41.18,Standard,37.90
1,AverageDailyPayFacTransactionExceedsAverageDai...,200.605000,292,35,40.63,Standard,34.81
5,IssuerIcardADPattern,0.000000,1,36,100.00,Standard,30.07
16,Transaction_amount,1232.446667,67,16,13.04,Standard,22.09
13,UniqueCardsRefusedLast24HoursThresholdBreached...,16.500000,6,32,71.43,Standard,22.02
11,UniqueCardsRefusedLast24HoursThresholdBreached,189.913333,10,42,60.00,Standard,21.30
14,WithdrawalRule,317.376667,77,32,40.12,Standard,21.00
7,LTVLowerThresholdBreached,232.725000,67,33,39.39,Deep,19.18
3,HighRiskIssuerRefusalReason,98.228333,10,29,50.00,Standard,16.85


### Scoring using general rule precision from our dashboard (added in rule_precision_L3M.sql file)
More in support of using this

In [123]:
# Create scoring system to determine optimal review type
# Factors: Loss risk, Precision, Volume

def calculate_priority_score_rule_precision_L3M(row):
    """
    Calculate priority score based on:
    - Monthly loss (higher = higher priority)
    - rule precision (higher = higher confidence, but also means we're catching them)
    - Volume (higher volume = more impact)
    """
    
    # Normalize loss (0-100 scale) so there is not too much skew from the highest loss rule and we can still differentiate between the lower loss rules
    max_loss = df_current['monthly_avg_fraud_volume_per_problematic_merchant'].max()  # Max average monthlyloss per problematic merchant
    loss_score = min((row['monthly_avg_fraud_volume_per_problematic_merchant'] / max_loss) * 100, 100) if max_loss > 0 else 0
    

    # rule precision score  L30D
    rule_precision_score  = round(row['rule_precision_L3M'],2)
    
    # effort required (volume * review time) score (normalized as well)
    max_effort = (df_current['Avg_Cases_Per_Month'] * df_current['Avg_Handling_Time_Minutes']).max()
    effort = row['Avg_Cases_Per_Month'] * row['Avg_Handling_Time_Minutes']
    volume_score = (effort / max_effort) * 100 if max_effort > 0 else 0
    
    
    # Weighted score: Loss (50%), Precision (30%), Volume (20%)
    priority_score_rule_precision_L30D = (loss_score * 0.5) + (rule_precision_score * 0.3) + (volume_score * 0.2)
    
    return priority_score_rule_precision_L30D 

df_current['Priority_Score'] = round(df_current.apply(calculate_priority_score_rule_precision_L3M, axis=1), 2)

# Sort by priority
df_current_sorted = df_current.sort_values(by = 'Priority_Score', ascending=False)

print("\nRULE PRIORITIZATION")
print("=" * 120)

df_current_sorted[['Rule_Name', 'monthly_avg_fraud_volume_per_problematic_merchant', 'Avg_Cases_Per_Month', 'Avg_Handling_Time_Minutes', 'rule_precision_L3M', 'Current_Review_Type', 'Priority_Score']]


RULE PRIORITIZATION


,Rule_Name,monthly_avg_fraud_volume_per_problematic_merchant,Avg_Cases_Per_Month,Avg_Handling_Time_Minutes,rule_precision_L3M,Current_Review_Type,Priority_Score
0,AttemptedAmountExceedsTransferAmount,3831.531667,23,20,67.82,Standard,71.25
1,AverageDailyPayFacTransactionExceedsAverageDai...,200.605000,292,35,74.13,Standard,44.86
9,PaymentForRequestRecentlyPublished,1860.913333,14,46,62.50,Standard,44.29
16,Transaction_amount,1232.446667,67,16,50.67,Standard,33.38
5,IssuerIcardADPattern,0.000000,1,36,100.00,Standard,30.07
11,UniqueCardsRefusedLast24HoursThresholdBreached,189.913333,10,42,84.62,Standard,28.69
8,PayerDisputeThreshold,0.000000,7,19,94.44,Standard,28.59
15,emailAgeScoreAboveThreshold,197.081667,13,23,83.87,Standard,28.32
12,UniqueCardsRefusedLast24HoursThresholdBreached...,153.685000,20,38,78.95,Standard,27.18
2,Bin6AttemptedVolumeSpike,293.658333,32,23,71.00,Standard,26.57


### New review type assignment
Using this score together with the current review type, we can assign a new review type.  
Rules with higer score get a better review.
Review priority flows as:
> **Light  → Standard → Deep**

Ideally, I like to keep **LTVHigherThresholdBreached** as **Deep** review due to the original reason this rule was created.

In [131]:
def rule_review_assignment(df, available_hours_weekly, fixed_rules=None):
    """
    Assigns review types to rules under a weekly capacity constraint.

    Review time per case:
      - Deep / Standard: use Avg_Handling_Time_Minutes (rule-specific)
      - Light         : fixed 15 minutes

    Constraints:
      - Proposed_Review_Type cannot be higher than Current_Review_Type
      - Capacity check (same as sheet):
          weekly_hours = ((total_mins / 60) + 24.8) * (12/52) + 20
        weekly_hours must be <= available_hours_weekly
    """
    if fixed_rules is None:
        fixed_rules = {}

    LIGHT_MINS = 15
    REVIEW_ORDER = ['Light', 'Standard', 'Deep']

    def level_index(t):
        """Map review type to numeric level: Light=0, Standard=1, Deep=2."""
        return REVIEW_ORDER.index(t)

    def calculate_row_mins(r):
        """
        Minutes per month for this rule given Proposed_Review_Type.
        - Light: uses LIGHT_MINS (15)
        - Standard/Deep: use Avg_Handling_Time_Minutes from the data
        """
        r_type = r['Proposed_Review_Type']
        if r_type == 'Light':
            per_case_mins = LIGHT_MINS
        else:
            per_case_mins = r['Avg_Handling_Time_Minutes']
        return r['Avg_Cases_Per_Month'] * per_case_mins

    # 1) Sort rules by Priority_Score (high → low),
    #    so we allocate capacity to most important rules first
    df = df_current.sort_values('Priority_Score', ascending=False).reset_index(drop=True)

    # 2) Use Current_Review_Type directly as the "ceiling" (no normalization needed)
    #    Assumes Current_Review_Type already contains: 'Light', 'Standard', 'Deep'
    df['Current_Review_Level'] = df['Current_Review_Type']

    # 3) Initial proposal:
    #    - If a rule is fixed in fixed_rules → use that type
    #    - Otherwise start at 'Light' (cheapest review level)
    df['Proposed_Review_Type'] = df['Rule_Name'].apply(
        lambda x: fixed_rules.get(x, 'Light')
    )

    # 4) Greedy upgrade loop: try Light → Standard → Deep
    for idx, row in df.iterrows():
        rule_name = row['Rule_Name']

        # Skip rules that are manually fixed
        if rule_name in fixed_rules:
            continue

        # This rule cannot go above its current review level
        current_max_type = df.at[idx, 'Current_Review_Level']

        # Try potential upgrades in this order
        for target_type in ['Standard', 'Deep']:
            current_proposed_type = df.at[idx, 'Proposed_Review_Type']

            # Only move upwards in effort/quality
            if level_index(target_type) <= level_index(current_proposed_type):
                continue

            # Enforce: do not exceed the current review level
            if level_index(target_type) > level_index(current_max_type):
                continue

            # Tentatively upgrade this rule
            df.at[idx, 'Proposed_Review_Type'] = target_type

            # Compute total monthly minutes for all rules
            total_mins_for_check = df.apply(calculate_row_mins, axis=1).sum()

            # Convert to weekly hours with your sheet formula:
            # ((total_mins / 60) + 24.8) * (12/52) + 20
            calculated_weekly_hours_for_check = (
                (total_mins_for_check / 60) + 24.8
            ) * (12/52) + 20

            # If we exceed capacity, revert this upgrade and stop upgrading this rule
            if calculated_weekly_hours_for_check > available_hours_weekly:
                df.at[idx, 'Proposed_Review_Type'] = current_proposed_type
                break

    # 5) After optimisation, compute time columns

    # Minutes per month, based on final Proposed_Review_Type
    df["Proposed_Time_Minutes"] = df.apply(calculate_row_mins, axis=1)
    # Hours per month
    df["Proposed_Time_Hours"] = (df["Proposed_Time_Minutes"] / 60.0).round(2)

    # Final total weekly hours, same formula as above
    final_total_mins = df["Proposed_Time_Minutes"].sum()
    final_calculated_weekly_hours = (
        (final_total_mins / 60) + 24.8
    ) * (12/52) + 20  #calculated in sheet cell H32

    print(f"Final Total Weekly Hours: {final_calculated_weekly_hours:.2f}")

    return df


target_weekly_cap = 75 #from sheet cell J3

locked_rules = {
    'LTVHigherThresholdBreached': 'Deep' #can fix other rules too like Lower LTV
}

df_final = rule_review_assignment(
    df_current,
    target_weekly_cap,
    fixed_rules=locked_rules
)

print("\nPROPOSED REVIEW TYPE")
print("=" * 120)

df_final[[
    "Rule_Name",
    "Current_Review_Type",
    "Priority_Score",
    "Proposed_Review_Type",
    "Avg_Cases_Per_Month",
    "Avg_Handling_Time_Minutes",
    "Proposed_Time_Hours"
]].sort_values('Rule_Name')

Final Total Weekly Hours: 74.92

PROPOSED REVIEW TYPE


,Rule_Name,Current_Review_Type,Priority_Score,Proposed_Review_Type,Avg_Cases_Per_Month,Avg_Handling_Time_Minutes,Proposed_Time_Hours
0,AttemptedAmountExceedsTransferAmount,Standard,71.25,Standard,23,20,7.67
1,AverageDailyPayFacTransactionExceedsAverageDai...,Standard,44.86,Light,292,35,73.00
9,Bin6AttemptedVolumeSpike,Standard,26.57,Light,32,23,8.00
12,HighRiskIssuerRefusalReason,Standard,22.62,Light,10,29,2.50
15,IssuerAndMerchantConcentration,Standard,14.67,Light,27,26,6.75
4,IssuerIcardADPattern,Standard,30.07,Standard,1,36,0.60
16,LTVHigherThresholdBreached,Deep,11.17,Deep,9,110,16.50
13,LTVLowerThresholdBreached,Deep,19.95,Light,67,33,16.75
6,PayerDisputeThreshold,Standard,28.59,Standard,7,19,2.22
2,PaymentForRequestRecentlyPublished,Standard,44.29,Standard,14,46,10.73
